In [0]:
%restart_python

In [0]:
!pip install pydeequ

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import os
os.environ["SPARK_VERSION"] = "3.3"
 
from pyspark.sql import SparkSession
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
os.environ['PYSPARK_SUBMIT_ARGS'] = '--conf spark.sql.shuffle.partitions=8 pyspark-shell'
 
# Initialize Spark session
spark = SparkSession.builder.appName("DataQualityChecks").getOrCreate()
 
# Define the schema for the custom DataFrame
schema = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("sales_amount", DoubleType(), True),
    StructField("transaction_date", StringType(), True)
])
 
# Sample data
data = [
    (1, 101, 250.0, "2023-01-01"),
    (2, 102, 100.5, "2023-01-02"),
    (3, 103, 450.25, "2023-01-03"),
    (4, 104, -50.0, "2023-01-04"),  # Invalid negative sales
    (5, 105, 320.75, "2023-01-05")
]
 
# Create the DataFrame
df = spark.createDataFrame(data, schema)
 
# Define the check (without any lambda for simplicity)
check = Check(spark, CheckLevel.Error, "Data quality checks") \
    .hasSize(lambda x: x >= 3) \
 
result = VerificationSuite(spark) \
    .onData(df) \
    .addCheck(check) \
    .run()
 
# Collect results as a dictionary
result_dict = result.checkResults
 
# Convert results to JSON-like format
import json
report_json = json.dumps(result_dict, indent=4)
 
# Print the report to check the results
print(report_json)
 
# Check if the result was successful
if result.status != "Success":
    print("Deequ checks failed.")
    raise SystemExit("Data quality checks failed")
 
print("Deequ checks passed")

[
    {
        "check_status": "Success",
        "check_level": "Error",
        "constraint_status": "Success",
        "check": "Data quality checks",
        "constraint_message": "",
        "constraint": "SizeConstraint(Size(None))"
    }
]
Deequ checks passed


In [0]:
import json

# Your Deequ results
result = [
    {
        "check_status": "Success",
        "check_level": "Error",
        "constraint_status": "Success",
        "check": "Data quality checks",
        "constraint_message": "",
        "constraint": "SizeConstraint(Size(None))"
    }
]

# Print in notebook (Databricks UI)
print(json.dumps(result, indent=4))
print("Deequ checks passed")

#dbfs:/FileStore/shared_uploads/traininguser5@sudosu.ai/quality_checks.yml
# Save to DBFS so GitHub Actions can read it
with open("/dbfs/FileStore/shared_uploads/traininguser5@sudosu.ai/deequ_result.json", "w") as f:
    json.dump({"result": result, "message": "Deequ checks passed"}, f)


[
    {
        "check_status": "Success",
        "check_level": "Error",
        "constraint_status": "Success",
        "check": "Data quality checks",
        "constraint_message": "",
        "constraint": "SizeConstraint(Size(None))"
    }
]
Deequ checks passed


In [0]:
import os
import json
from pyspark.sql import SparkSession
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Environment setup
os.environ["SPARK_VERSION"] = "3.3"
os.environ['PYSPARK_SUBMIT_ARGS'] = '--conf spark.sql.shuffle.partitions=8 pyspark-shell'

# Initialize Spark session
spark = SparkSession.builder.appName("DataQualityChecks").getOrCreate()

# Schema
schema = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("sales_amount", DoubleType(), True),
    StructField("transaction_date", StringType(), True)
])

# Sample data
data = [
    (1, 101, 250.0, "2023-01-01"),
    (2, 102, 100.5, "2023-01-02"),
    (3, 103, 450.25, "2023-01-03"),
    (4, 104, -50.0, "2023-01-04"),  # Invalid negative sales
    (5, 105, 320.75, "2023-01-05")
]

df = spark.createDataFrame(data, schema)

# Data quality check
check = Check(spark, CheckLevel.Error, "Data quality checks") \
    .hasSize(lambda x: x >= 3)

result = VerificationSuite(spark) \
    .onData(df) \
    .addCheck(check) \
    .run()

# Convert to JSON
result_dict = result.checkResults
report_json = json.dumps(result_dict, indent=4)

# --- 1️ Print in Databricks notebook UI ---
print("=== Deequ Report ===")
print(report_json)

# --- 2️ Save to DBFS so GitHub Actions can read it ---
dbfs_path = "/dbfs/FileStore/shared_uploads/traininguser5@sudosu.ai/deequ_result.json"
with open(dbfs_path, "w") as f:
    json.dump({"result": result_dict, "status": result.status}, f, indent=4)

print(f"Deequ report saved to: {dbfs_path}")

if result.status != "Success":
    raise SystemExit("Data quality checks failed")

print("Deequ checks passed ")


=== Deequ Report ===
[
    {
        "check_status": "Success",
        "check_level": "Error",
        "constraint_status": "Success",
        "check": "Data quality checks",
        "constraint_message": "",
        "constraint": "SizeConstraint(Size(None))"
    }
]
Deequ report saved to: /dbfs/FileStore/shared_uploads/traininguser5@sudosu.ai/deequ_result.json
Deequ checks passed 
